# Alternative Entropy Analysis

This notebook loads the synthetic dataset produced by `dataset_create_and_explore.ipynb` and computes entropy using two approaches that differ from the `scipy.stats.entropy` method used there:

1. **Pure-Python Shannon entropy** — using the custom `entropy.py` module in this repo (Counter-based, `math.log2`)
2. **Rényi entropy** — a generalisation of Shannon entropy parameterised by order `α`; at `α → 1` it converges to Shannon entropy

In [1]:
print("hello")

hello


## 1. Load Dataset

In [2]:
import pandas as pd

df = pd.read_csv("synthetic_dataset.csv")
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Loaded dataset: 1000 rows × 6 columns


,text_message,url,language,country,device,star_review
0,"Exactly as described, fast delivery.",https://www.mystore.co/review/789,Spanish,Germany,Mobile,4
1,"Great product, really happy with it!",https://www.quickbuy.biz/item/456,Spanish,China,Desktop,5
2,Decent value for the price.,https://www.quickbuy.biz/listing/654,German,Brazil,Mobile,5
3,The item looks different from the photo.,https://www.quickbuy.biz/review/789,English,France,Mobile,4
4,The item looks different from the photo.,https://www.buynow.shop/listing/654,English,United Kingdom,Tablet,3


## 2. Column Classification

In [5]:
all_columns = list(df.columns)
print("Columns:", all_columns)

column_classification = {
    "semi_identifiers": ['url','language', 'country', 'device', 'star_review'],  #  fill in
    "sensitive":        ['text_message'],  # fill in
}

print("Semi-identifiers:", column_classification["semi_identifiers"])
print("Sensitive:       ", column_classification["sensitive"])

Columns: ['text_message', 'url', 'language', 'country', 'device', 'star_review']
Semi-identifiers: ['url', 'language', 'country', 'device', 'star_review']
Sensitive:        ['text_message']


I-Entity Dataset

The I-entity dataset is formed by dropping the sensitive attribute(s), retaining only identifiers and semi-identifiers.

In [8]:
df_i_entity = df[column_classification["semi_identifiers"]].copy()

print(f"Original dataset : {df.shape}")
print(f"I-entity dataset : {df_i_entity.shape}")
print(f"Dropped columns  : {column_classification['sensitive']}")
df_i_entity.head()

Original dataset : (1000, 6)
I-entity dataset : (1000, 5)
Dropped columns  : ['text_message']


,url,language,country,device,star_review
0,https://www.mystore.co/review/789,Spanish,Germany,Mobile,4
1,https://www.quickbuy.biz/item/456,Spanish,China,Desktop,5
2,https://www.quickbuy.biz/listing/654,German,Brazil,Mobile,5
3,https://www.quickbuy.biz/review/789,English,France,Mobile,4
4,https://www.buynow.shop/listing/654,English,United Kingdom,Tablet,3


## 4. Matching Probabilities

Group the S-entity dataset by quasi-identifiers (semi-identifiers) to form **equivalence classes** — sets of records that are indistinguishable on those attributes.

For a target I-entity, the **matching probability** is the chance an attacker correctly links one record in its equivalence class to that individual:

$$P_{match} = \frac{1}{|equivalence\ class|}$$

A smaller class → higher matching probability → greater re-identification risk.

In [9]:
quasi_ids = column_classification["semi_identifiers"]

# Size of each equivalence class
equiv_classes = (
    df_i_entity
    .groupby(quasi_ids)
    .size()
    .reset_index(name="class_size")
)
equiv_classes["match_probability"] = 1 / equiv_classes["class_size"]

equiv_classes = equiv_classes.sort_values("class_size").reset_index(drop=True)

print(f"Total equivalence classes : {len(equiv_classes)}")
print(f"Smallest class size       : {equiv_classes['class_size'].min()}")
print(f"Largest class size        : {equiv_classes['class_size'].max()}")
print(f"Mean class size           : {equiv_classes['class_size'].mean():.2f}")
equiv_classes.head(10)

Total equivalence classes : 956
Smallest class size       : 1
Largest class size        : 3
Mean class size           : 1.05


,url,language,country,device,star_review,class_size,match_probability
0,https://www.buynow.shop/item/456,Arabic,China,Desktop,5,1,1.0
1,https://www.rateit.app/listing/654,French,France,Mobile,2,1,1.0
2,https://www.rateit.app/listing/654,French,France,Mobile,4,1,1.0
3,https://www.rateit.app/listing/654,French,United States,Desktop,3,1,1.0
4,https://www.rateit.app/listing/654,German,India,Desktop,5,1,1.0
5,https://www.rateit.app/listing/654,Mandarin,Mexico,Desktop,2,1,1.0
6,https://www.rateit.app/listing/654,Mandarin,Mexico,Desktop,3,1,1.0
7,https://www.rateit.app/listing/654,Mandarin,Mexico,Mobile,3,1,1.0
8,https://www.rateit.app/listing/654,Mandarin,Mexico,Mobile,4,1,1.0
9,https://www.rateit.app/listing/654,Mandarin,United States,Mobile,1,1,1.0


In [ ]:
def matching_probabilities(
    s_df: pd.DataFrame,
    i_df: pd.DataFrame,
    quasi_ids: list[str],
    sensitive_col: str,
) -> pd.DataFrame:
    """Calculate P(S_i, I_j) for every I-entity / sensitive-value pair.

    For each record I_j in i_df, finds the matching equivalence class in s_df
    (records sharing the same quasi-identifier values) and computes the
    probability of each sensitive attribute value within that class.

    Args:
        s_df:          S-entity dataset — contains quasi-identifiers and the
                       sensitive attribute column.
        i_df:          I-entity dataset — contains only quasi-identifier columns.
        quasi_ids:     Column names to group on (quasi-identifiers / semi-identifiers).
        sensitive_col: Name of the sensitive attribute column in s_df.

    Returns:
        DataFrame with columns:
            i_entity_index   — original row index from i_df
            <quasi_id cols>  — the quasi-identifier values for that I-entity
            sensitive_value  — a distinct value of the sensitive attribute
            match_count      — how many S-entity records in the group have this value
            group_size       — total records in the matched equivalence class
            probability      — match_count / group_size  (i.e. P(S_i, I_j))
        Unmatched I-entities (no group in s_df) are included with NaN probability.
    """
    # Pre-compute frequency distribution for every equivalence class in s_df
    group_counts = (
        s_df.groupby(quasi_ids + [sensitive_col])
        .size()
        .reset_index(name="match_count")
    )
    group_totals = (
        s_df.groupby(quasi_ids)
        .size()
        .reset_index(name="group_size")
    )
    group_dist = group_counts.merge(group_totals, on=quasi_ids)
    group_dist["probability"] = group_dist["match_count"] / group_dist["group_size"]

    # Tag each I-entity with its original index, then merge on quasi-identifiers
    i_tagged = i_df[quasi_ids].copy()
    i_tagged.insert(0, "i_entity_index", i_tagged.index)

    result = i_tagged.merge(group_dist, on=quasi_ids, how="left")
    result = result.rename(columns={sensitive_col: "sensitive_value"})

    return result.reset_index(drop=True)


# --- Example call (fill in column names once classification is complete) ---
sensitive_col = column_classification["sensitive"][0] if column_classification["sensitive"] else None

if quasi_ids and sensitive_col:
    prob_df = matching_probabilities(df, df_i_entity, quasi_ids, sensitive_col)
    print(f"Result shape: {prob_df.shape}")
    print(prob_df.head(20).to_string(index=False))
else:
    print("Fill in semi_identifiers and sensitive in column_classification to run this cell.")